# 01 - Quickstart Tutorial: polars-anta

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huuan26/polars-anta/blob/master/examples/01_quickstart.ipynb)

Welcome to **polars-anta**! This notebook demonstrates how to calculate technical indicators and statistical transforms natively using Polars expressions (`pl.Expr`).


In [ ]:
# Install polars-anta (uncomment if running in Colab)
!pip install -q polars polars-anta matplotlib


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import polars_anta  # Registers the .ta namespace

print(f"Polars version: {pl.__version__}")
print(f"polars-anta version: {polars_anta.__version__}")
print(f"Available indicators: {len(polars_anta.__all__)}")

## 1. Create Sample Market Data (1,000 Bars)


In [ ]:
np.random.seed(42)
n = 1000

returns = np.random.normal(0.0002, 0.015, n)
close = 100.0 * np.exp(np.cumsum(returns))
high = close * (1.0 + np.abs(np.random.normal(0, 0.005, n)))
low = close * (1.0 - np.abs(np.random.normal(0, 0.005, n)))
open_ = (high + low) / 2.0
volume = np.random.randint(1000, 50000, n).astype(float)

df = pl.DataFrame(
    {
        "open": open_,
        "high": high,
        "low": low,
        "close": close,
        "volume": volume,
    }
)

print(df.head())

## 2. Calculating Indicators with the `.ta` Namespace


In [ ]:
# Compute Moving Averages, RSI, and Z-Score in a single expression pass
features = df.with_columns(
    pl.col("close").ta.sma(20).alias("sma_20"),
    pl.col("close").ta.ema(20).alias("ema_20"),
    pl.col("close").ta.rsi(14).alias("rsi_14"),
    pl.col("close").ta.zscore(20).alias("zscore_20"),
    pl.col("close").ta.linear_regression_slope(14).alias("slope_14"),
)

print(features.select(["close", "sma_20", "rsi_14", "zscore_20", "slope_14"]).tail())

## 3. Working with Multi-Output Indicators (Struct Expressions)

Indicators like Bollinger Bands, MACD, and Ichimoku return typed `pl.Struct` expressions. Unnest them cleanly with `.unnest()`:


In [ ]:
df_bands = df.with_columns(
    pl.col("close").ta.bollinger_bands(period=20, num_std=2.0).alias("bb"),
    pl.col("close").ta.macd().alias("macd_struct"),
).unnest(["bb", "macd_struct"])

print(df_bands.select(["close", "upper", "middle", "lower", "macd", "signal", "histogram"]).tail())

## 4. Visualizing Results


In [ ]:
plot_df = df_bands.tail(150).to_pandas()

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2.5, 1]}
)

# Price & Bollinger Bands
ax1.plot(plot_df.index, plot_df["close"], label="Close Price", color="#1f77b4", lw=1.5)
ax1.plot(plot_df.index, plot_df["middle"], label="SMA (20)", color="#ff7f0e", linestyle="--", lw=1)
ax1.plot(plot_df.index, plot_df["upper"], label="Upper BB", color="#2ca02c", alpha=0.7, lw=1)
ax1.plot(plot_df.index, plot_df["lower"], label="Lower BB", color="#d62728", alpha=0.7, lw=1)
ax1.fill_between(plot_df.index, plot_df["lower"], plot_df["upper"], color="#2ca02c", alpha=0.1)
ax1.set_title("polars-anta: Bollinger Bands (20, 2) Analysis", fontsize=13, fontweight="bold")
ax1.set_ylabel("Price")
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)

# MACD
ax2.plot(plot_df.index, plot_df["macd"], label="MACD", color="#1f77b4", lw=1.2)
ax2.plot(plot_df.index, plot_df["signal"], label="Signal", color="#ff7f0e", lw=1.2)
colors = ["#2ca02c" if h >= 0 else "#d62728" for h in plot_df["histogram"]]
ax2.bar(plot_df.index, plot_df["histogram"], label="Hist", color=colors, alpha=0.5, width=0.8)
ax2.set_ylabel("MACD")
ax2.grid(True, alpha=0.3)
ax2.legend(loc="upper left")

plt.tight_layout()
plt.show()